In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr
from scipy.io import savemat, loadmat
from tqdm import tqdm
import seaborn as sns
import pickle
from support import normalise, pair_mutual_information, surrogate, total_mutual_information, task_producer
from corrector import Corrector
from correctorCMIknn import Corrector as knn_Corrector
import os

sns.set_theme("talk", "ticks")


# Designing a suitable distribution

In [ ]:
def _rot_mat (theta, B):
    _theta = np.array(theta)
    return np.moveaxis(np.array([[np.cos(_theta), B*np.sin(_theta)],[-np.sin(_theta)/B, np.cos(_theta)]]),2,0)
def rotation (vec:np.ndarray, theta:float, B:float=1):
    """
    Rotates vector on the plane of an angle theta around an ellipsis with y-oriented axis B times the x-oriented one.
    Input are expected to be np.ndarrays of any shape as long as at leas one dimension ha length two. If more than one
    dimension has length 2, the first is considered to be the spatial dimension.
    """
    if 2 in vec.shape:
        myAx = np.min(np.nonzero(np.array(vec.shape)==2))
        _vec = np.moveaxis(vec, myAx, 0)
        myShape = _vec.shape
        _vec = _vec.reshape((2,-1))
        if np.isscalar(theta):
            rot_mat = _rot_mat(np.full(_vec.shape[1], theta), B)
        else:
            rot_mat = _rot_mat(theta, B)
        res = np.empty_like(_vec)
        for i in range(_vec.shape[1]):
            res[:,i]=_vec[:,i]@rot_mat[i]
        return np.moveaxis(res.reshape(myShape),0,myAx)
    else:
        raise ValueError(f"Invalid input shape {vec.shape}")


In [ ]:
fig, ax = plt.subplots(6,6, sharex="col", sharey="col", figsize=(12,12), squeeze=False)
N = 20000
for mf in range(2,8):
    for a, r in enumerate([0.]+[0.1+0.2*i for i in range(5)]):
        B = np.sqrt((1-r)/(1+r))
        x = np.zeros((2,N))
        x[0,:] = np.sqrt(np.random.chisquare(mf,N))
        x = rotation(rotation(x, 2*np.pi*np.random.random(x.size//2),B),np.pi/4)#0)#
        lab = r"$\rho$"+f": {pearsonr(*x)[0]:.2}"
        plt.sca(ax[a,mf-2])
        plt.hist2d(*x, bins=40, density=True)
        #ax[a,mf-2].set_aspect('equal', 'box')
        plt.text(-3,2,lab,color="lightgreen", size="small")
        if a==0:
            plt.title(f"df: {mf}")
        if mf>2:
            ax[a,mf-2].set_yticks([])
        if mf==7:
            ax2 = plt.twinx()
            ax2.set_yticks([])
            ax2.set_ylabel(f"r: {round(r,1)}")
plt.subplots_adjust(wspace=0.0, hspace=0.0)
plt.show()

## Creating fake distributions

In [ ]:
P = 1000
t = tqdm(total=36, desc="Sym")
for N in [30000]:
    for df in range(2,8):
        for r in [0.]+[0.1+0.2*i for i in range(5)]:
            t.write(f"/mnt/DATA/NonLinearMI/verify/crafted_{N}_{df}_r{f'{r:2}'[2]}.mat")
            if os.path.isfile(f"/mnt/DATA/NonLinearMI/verify/crafted_{N}_{df}_r{f'{r:2}'[2]}.mat"):
                t.update()
                continue
            B = np.sqrt((1-r)/(1+r))
            exp = np.zeros((N,2,P))
            exp[:,0,:] = np.sqrt(np.random.chisquare(df,(N,P)))
            exp = rotation(rotation(exp, 2*np.pi*np.random.random(N*P),B),np.pi/4)
            # print(f"/mnt/DATA/NonLinearMI/verify/crafted_{N}_{df}_r{f'{r:2}'[2]}")
            savemat(f"/mnt/DATA/NonLinearMI/verify/crafted_{N}_{df}_r{f'{r:2}'[2]}.mat",{"crafted":exp})
            t.update()
t.close()

# Binning method
## Loading results

In [ ]:
if os.path.isfile("verify30-10000_nc.pickle"):
    with open("verify30-10000_nc.pickle", "rb") as fp:
        previous = pickle.load(fp)
    values = {int(N):{int(df):{int(r):{int(bin):previous["values"][N][df][r][bin] for bin in previous["values"][N][df][r]} for r in previous["values"][N][df]} for df in previous["values"][N]} for N in previous["values"]}
    shadow = {int(N):{int(df):{int(r):{int(bin):previous["shadow"][N][df][r][bin] for bin in previous["shadow"][N][df][r]} for r in previous["shadow"][N][df]} for df in previous["shadow"][N]} for N in previous["shadow"]}
    correl = {int(N):{int(df):{int(r):{int(bin):previous["correl"][N][df][r][bin] for bin in previous["correl"][N][df][r]} for r in previous["correl"][N][df]} for df in previous["correl"][N]} for N in previous["correl"]}
    del previous
    print(len(values))
else:
    values = {}
    shadow = {}
    correl = {}
tot = 6*6*6*5*1000

with tqdm(total=tot, desc="Loading") as t:
    for N in [30, 100, 300, 1000, 3000, 10000, 30000]:
        if N in values:
            t.update(6*6*5*1000)
            continue
        values[N]={}
        shadow[N]={}
        correl[N]={}
        for df in range(2,8):
            values[N][df]={}
            shadow[N][df]={}
            correl[N][df]={}
            for r in [0,1,3,5,7,9]:
                values[N][df][r]={}
                shadow[N][df][r]={}
                correl[N][df][r]={}
                for bin in [4,8,16,32,64]:
                    values[N][df][r][bin]=np.zeros((1000,100))
                    shadow[N][df][r][bin]=np.zeros((1000,100))
                    correl[N][df][r][bin]=np.zeros(1000)
                    for pn in range(1000):
                        values[N][df][r][bin][pn,:]=np.load(f"/mnt/DATA/NonLinearMI/crafted_30000_{df}_r{r}_{N}_bin{bin}/patient{pn:02}_{bin}.npy")
                        shadow[N][df][r][bin][pn,:]=np.load(f"/mnt/DATA/NonLinearMI/crafted_30000_{df}_r{r}_{N}_bin{bin}/patient{pn:02}_{bin}_sha.npy")
                        correl[N][df][r][bin][pn]=np.load(f"/mnt/DATA/NonLinearMI/crafted_30000_{df}_r{r}_{N}_bin{bin}/patient{pn:02}_{bin}_cor.npy")
                        t.update()
                  
if not os.path.isfile("verify30-30000_nc.pickle"):
    with open("verify30-30000_nc.pickle","wb") as fp:
        pickle.dump({"values":values,"shadow":shadow,"correl":correl},fp)

### Correcting

In [ ]:
if os.path.isfile("verify30-10000.pickle"):
    with open("verify30-10000.pickle", "rb") as fp:
        previous = pickle.load(fp)
    
    values_c = {int(N):{int(df):{int(r):{int(bin):previous["values"][N][df][r][bin] for bin in previous["values"][N][df][r]} for r in previous["values"][N][df]} for df in previous["values"][N]} for N in previous["values"]}
    shadow_c = {int(N):{int(df):{int(r):{int(bin):previous["shadow"][N][df][r][bin] for bin in previous["shadow"][N][df][r]} for r in previous["shadow"][N][df]} for df in previous["shadow"][N]} for N in previous["shadow"]}
    correl = {int(N):{int(df):{int(r):{int(bin):previous["correl"][N][df][r][bin] for bin in previous["correl"][N][df][r]} for r in previous["correl"][N][df]} for df in previous["correl"][N]} for N in previous["correl"]}
    del previous
else:
    values_c = {}
    shadow_c = {}
    correl = {}
tot = 6*6*5*6

with tqdm(total=tot, desc="Correcting") as t:
    for N in values:
        if N in values_c:
            t.update(6*6*5)
            continue
        values_c[N]={}
        shadow_c[N]={}
        for df in values[N]:
            values_c[N][df]={}
            shadow_c[N][df]={}    
            for r in values[N][df]:
                values_c[N][df][r]={}
                shadow_c[N][df][r]={}        
                for bin in shadow[N][df][r]:
                    corrector = Corrector(200, f"/mnt/DATA/NonLinearMI/crafted_30000_2_r0_{N}_bin{bin}",30000, N, bin)
                    corrector.compute_correction()
                    values_c[N][df][r][bin]=corrector.correctI(values[N][df][r][bin])
                    shadow_c[N][df][r][bin]=corrector.correctI(shadow[N][df][r][bin])
                    t.update()
if not os.path.isfile("verify30-30000.pickle"):
    with open("verify30-30000.pickle","wb") as fp:
        pickle.dump({"values":values_c,"shadow":shadow_c,"correl":correl},fp)

## Display results

In [ ]:
with open("verify30-30000.pickle", "rb") as fp:
    previous = pickle.load(fp)
    values_c = {int(N):{int(df):{int(r):{int(bin):previous["values"][N][df][r][bin] for bin in previous["values"][N][df][r]} for r in previous["values"][N][df]} for df in previous["values"][N]} for N in previous["values"]}
    shadow_c = {int(N):{int(df):{int(r):{int(bin):previous["shadow"][N][df][r][bin] for bin in previous["shadow"][N][df][r]} for r in previous["shadow"][N][df]} for df in previous["shadow"][N]} for N in previous["shadow"]}
    correl = {int(N):{int(df):{int(r):{int(bin):previous["correl"][N][df][r][bin] for bin in previous["correl"][N][df][r]} for r in previous["correl"][N][df]} for df in previous["correl"][N]} for N in previous["correl"]}
    del previous

In [ ]:
fig, ax = plt.subplots(6,6, sharex=True, sharey=True, figsize=(16,16), squeeze=False)
for i,r in enumerate([0,1,3,5,7,9]):
    for j,df in enumerate(range(2,8)):
        plt.sca(ax[i,j])
        for bin in [4,8,16,32,64]:
            x = []
            y = []
            yu = []
            yd = []
            for N in [30, 100, 300, 1000, 3000, 10000, 30000]:
                x.append(N)
                tmp = values_c[N][df][r][bin]
                # tmp = shadow_c[N][df][r][bin]
                tmp2 = 2*(tmp[:,0]-np.mean(tmp[:,1:],1))/(tmp[:,0]+np.mean(tmp[:,1:],1))#tmp[:,0]-np.mean(tmp[:,1:],1)
                d,m,u = np.quantile(tmp2,[.25,.50,.75])
                y.append(m)
                yu.append(u-m)
                yd.append(m-d)
            p=plt.errorbar(x,y,[yu,yd],fmt="-^", elinewidth=1.5, capsize=3, label=bin)
        plt.plot([30,30000], [0,0],":k",lw=2,zorder=3)
        plt.xlabel("N")
        if i==0:
            plt.title(f"df: {df}")
        # if i==5:
        if (i,j)==(3,5):
            plt.legend(title="# bins", loc='center left', bbox_to_anchor=(1.2, 1))
        if j==0:
            plt.ylabel(r"$\Delta$TMI")
        if j==5:
            ax2 = plt.twinx()
            ax2.set_yticks([])
            ax2.set_ylabel(f"r: 0.{r}")

ax[0,0].set_ylim((-.2,2.1))
plt.xscale("log")
plt.subplots_adjust(wspace=0.0, hspace=0.0)
plt.show()


In [ ]:
fig, ax = plt.subplots(6,7, sharex=True, sharey=True, figsize=(16,16), squeeze=False)
for i,r in enumerate([0,1,3,5,7,9]):
    for j,N in enumerate([30, 100, 300, 1000, 3000, 10000, 30000]):
        plt.sca(ax[i,j])
        for bin in [4,8,16,32,64]:
            x = []
            y = []
            yu = []
            yd = []
            for df in range(2,8):
                x.append(df)
                tmp = values_c[N][df][r][bin]
                # tmp = shadow_c[N][df][r][bin]
                tmp2 = (tmp[:,0]-np.mean(tmp[:,1:],1))/(tmp[:,0]+np.mean(tmp[:,1:],1))
                d,m,u = np.quantile(tmp2,[.25,.50,.75])
                y.append(m)
                yu.append(u-m)
                yd.append(m-d)
            p=plt.errorbar(x,y,[yu,yd],fmt="-^", elinewidth=1.5, capsize=3, label=bin, alpha=1 if bin<30 else 0.4)
        plt.plot([2,7], [0,0],":k",lw=2,zorder=3)
        if i==5:
            plt.xlabel("df")
            plt.xticks([2, 4, 6])
        if i==0:
            plt.title(f"N: {N}")
        # if i==5:
        if (i,j)==(3,6):
            plt.legend(title="# bins", loc='center left', bbox_to_anchor=(1.2, 1))
        if j==0:
            plt.ylabel(r"$\Delta$TMI")
        if j==6:
            ax2 = plt.twinx()
            ax2.set_yticks([])
            ax2.set_ylabel(f"r: 0.{r}")
# ax[0,0].set_ylim((-.2,0.2))
ax[0,0].set_ylim((-.2,2.1))
plt.subplots_adjust(wspace=0.0, hspace=0.0)
plt.savefig("bias.pdf")
plt.show()


In [ ]:
N=1000
r=1
for bin in [4,8,16,32,64]:
    x = []
    y = []
    yu = []
    yd = []
    for df in range(2,8):
        x.append(df)
        tmp = values_c[N][df][r][bin]
        # tmp = shadow_c[N][df][r][bin]
        tmp2 = (tmp[:,0]-np.mean(tmp[:,1:],1))/(tmp[:,0]+np.mean(tmp[:,1:],1))
        d,m,u = np.quantile(tmp2,[.25,.50,.75])
        y.append(m)
        yu.append(u-m)
        yd.append(m-d)
    p=plt.errorbar(x,y,[yu,yd],fmt="-^", elinewidth=1.5, capsize=3, label=bin, alpha=1 if bin>4 else 0.4)
plt.plot([2,7], [0,0],":k",lw=2,zorder=3)
plt.legend(ncol=2, title="# bins")
plt.title("N = 1000, r = 0.1")
plt.xlabel("df")
plt.ylabel(r"$\Delta$TMI")
plt.show()

# KNN method
## Loading results

In [ ]:
if os.path.isfile("knn_verify30-30000_nc.pickle"):
    with open("knn_verify30-30000_nc.pickle", "rb") as fp:
        previous = pickle.load(fp)
    knn_values = {int(N):{int(df):{int(r):{int(bin):previous["values"][N][df][r][bin] for bin in previous["values"][N][df][r]} for r in previous["values"][N][df]} for df in previous["values"][N]} for N in previous["values"]}
    knn_shadow = {int(N):{int(df):{int(r):{int(bin):previous["shadow"][N][df][r][bin] for bin in previous["shadow"][N][df][r]} for r in previous["shadow"][N][df]} for df in previous["shadow"][N]} for N in previous["shadow"]}
    knn_correl = {int(N):{int(df):{int(r):{int(bin):previous["correl"][N][df][r][bin] for bin in previous["correl"][N][df][r]} for r in previous["correl"][N][df]} for df in previous["correl"][N]} for N in previous["correl"]}
    del previous
    print(len(knn_values))
else:
    knn_values = {}
    knn_shadow = {}
    knn_correl = {}
tot = 6*6*6*5*1000

with tqdm(total=tot, desc="Loading") as t:
    for N in [30, 100, 300, 1000, 3000, 10000]:
        if not N in knn_values:
            knn_values[N]={}
            knn_shadow[N]={}
            knn_correl[N]={}
        for df in range(2,8):
            if not df in knn_values[N]:
                knn_values[N][df]={}
                knn_shadow[N][df]={}
                knn_correl[N][df]={}
            for r in [0,1,3,5,7,9]:
                if not r in knn_values[N][df]:
                    knn_values[N][df][r]={}
                    knn_shadow[N][df][r]={}
                    knn_correl[N][df][r]={}
                for bin in [2,6,18,54,162]:
                    if not bin in knn_values[N][df][r]:
                        knn_values[N][df][r][bin]=np.full((1000,100), np.nan)
                        knn_shadow[N][df][r][bin]=np.full((1000,100), np.nan)
                        knn_correl[N][df][r][bin]=np.full(1000, np.nan)
                    if not np.isnan(knn_correl[N][df][r][bin][999]):
                        t.update(1000)
                        continue
                    for pn in range(1000):
                        if np.isnan(knn_values[N][df][r][bin][pn,0]) and os.path.isfile(f"/mnt/DATA/NonLinearMI/knn_crafted_30000_{df}_r{r}_{N}_bin{bin}/knn_patient{pn:02}_{bin}_cor.npy"):
                            knn_values[N][df][r][bin][pn,:]=np.load(f"/mnt/DATA/NonLinearMI/knn_crafted_30000_{df}_r{r}_{N}_bin{bin}/knn_patient{pn:02}_{bin}.npy")
                            knn_shadow[N][df][r][bin][pn,:]=np.load(f"/mnt/DATA/NonLinearMI/knn_crafted_30000_{df}_r{r}_{N}_bin{bin}/knn_patient{pn:02}_{bin}_sha.npy")
                            knn_correl[N][df][r][bin][pn]=np.load(f"/mnt/DATA/NonLinearMI/knn_crafted_30000_{df}_r{r}_{N}_bin{bin}/knn_patient{pn:02}_{bin}_cor.npy")
                        t.update()
               
if not os.path.isfile("knn_verify30-30000_nc.pickle"):
    with open("knn_verify30-30000_nc.pickle","wb") as fp:
        pickle.dump({"values":knn_values,"shadow":knn_shadow,"correl":knn_correl},fp)

In [ ]:
print("N:",set(knn_values.keys()))
olk0 = set()
olk1 = set()
olk2 = set()
olk3 = set()
for N in knn_values:
    nek0 = set(knn_values[N].keys())
    if nek0 != olk0:
        print("df:",nek0)
        olk0 = nek0
    for df in knn_values[N]:
        nek1 = set(knn_values[N][df].keys())
        if nek1 != olk1:
            print("r:",nek1)
            olk1 = nek1
        for r in knn_values[N][df]:
            nek2 = set(knn_values[N][df][r].keys())
            if nek2 != olk2:
                print("bin:",nek2,N,df,r)
                olk2 = nek2
            for bin in knn_values[N][df][r]:
                if np.isnan(knn_values[N][df][r][bin][0,0]):
                    print(".", end="")

### Correcting

In [ ]:
if False and os.path.isfile("knn_verify30-30000.pickle"):
    with open("knn_verify30-30000.pickle", "rb") as fp:
        previous = pickle.load(fp)
    
    knn_values_c = {int(N):{int(df):{int(r):{int(bin):previous["values"][N][df][r][bin] for bin in previous["values"][N][df][r]} for r in previous["values"][N][df]} for df in previous["values"][N]} for N in previous["values"]}
    knn_shadow_c = {int(N):{int(df):{int(r):{int(bin):previous["shadow"][N][df][r][bin] for bin in previous["shadow"][N][df][r]} for r in previous["shadow"][N][df]} for df in previous["shadow"][N]} for N in previous["shadow"]}
    knn_correl = {int(N):{int(df):{int(r):{int(bin):previous["correl"][N][df][r][bin] for bin in previous["correl"][N][df][r]} for r in previous["correl"][N][df]} for df in previous["correl"][N]} for N in previous["correl"]}
    del previous
else:
    knn_values_c = {}
    knn_shadow_c = {}
    knn_correl = {}
tot = 6*6*6*5

with tqdm(total=tot, desc="Correcting") as t:
    for N in knn_values:
        if N in knn_values_c:
            t.update(6*6*5)
            continue
        knn_values_c[N]={}
        knn_shadow_c[N]={}
        for df in knn_values[N]:
            knn_values_c[N][df]={}
            knn_shadow_c[N][df]={}    
            for r in knn_values[N][df]:
                knn_values_c[N][df][r]={}
                knn_shadow_c[N][df][r]={}        
                for bin in knn_shadow[N][df][r]:
                    if np.isnan(knn_shadow[N][df][r][bin][0,0]):
                        knn_values_c[N][df][r][bin]=np.full_like(knn_values[N][df][r][bin], np.nan)
                        knn_shadow_c[N][df][r][bin]=np.full_like(knn_shadow[N][df][r][bin], np.nan)
                    else:
                        corrector = knn_Corrector(200, f"/mnt/DATA/NonLinearMI/knn_crafted_30000_2_r0_{N}_bin{bin}",30000, N, bin)
                        corrector.compute_correction()
                        knn_values_c[N][df][r][bin]=corrector.correctI(knn_values[N][df][r][bin])
                        knn_shadow_c[N][df][r][bin]=corrector.correctI(knn_shadow[N][df][r][bin])
                    t.update()

if not os.path.isfile("knn_verify30-30000.pickle"):
    with open("knn_verify30-30000.pickle","wb") as fp:
        pickle.dump({"values":knn_values_c,"shadow":knn_shadow_c,"correl":knn_correl},fp)

## Display results
### Non corrected

In [ ]:
with open("knn_verify30-30000_nc.pickle", "rb") as fp:
    previous = pickle.load(fp)
    knn_values_c = {int(N):{int(df):{int(r):{int(bin):previous["values"][N][df][r][bin] for bin in previous["values"][N][df][r]} for r in previous["values"][N][df]} for df in previous["values"][N]} for N in previous["values"]}
    knn_shadow_c = {int(N):{int(df):{int(r):{int(bin):previous["shadow"][N][df][r][bin] for bin in previous["shadow"][N][df][r]} for r in previous["shadow"][N][df]} for df in previous["shadow"][N]} for N in previous["shadow"]}
    knn_correl = {int(N):{int(df):{int(r):{int(bin):previous["correl"][N][df][r][bin] for bin in previous["correl"][N][df][r]} for r in previous["correl"][N][df]} for df in previous["correl"][N]} for N in previous["correl"]}
    del previous

In [ ]:
fig, ax = plt.subplots(6,6, sharex=True, sharey=True, figsize=(16,16), squeeze=False)
for i,r in enumerate([0,1,3,5,7,9]):
    for j,df in enumerate(range(2,8)):
        plt.sca(ax[i,j])
        for bin in [2,6,18,54,162]:
            x = []
            y = []
            yu = []
            yd = []
            for N in [30, 100, 300, 1000, 3000, 10000]:
                x.append(N)
                tmp = knn_values[N][df][r][bin]
                # tmp = shadow_c[N][df][r][bin]
                tmp2 = (tmp[:,0]-np.mean(tmp[:,1:],1))/(tmp[:,0]+1e-4)#tmp[:,0]-np.mean(tmp[:,1:],1)
                d,m,u = np.quantile(tmp2,[.25,.50,.75])
                y.append(m)
                yu.append(u-m)
                yd.append(m-d)
            p=plt.errorbar(x,y,[yu,yd],fmt="-^", elinewidth=1.5, capsize=3, label=bin)
        plt.plot([30,10000], [0,0],":k",lw=2,zorder=3)
        plt.xlabel("N")
        if i==0:
            plt.title(f"df: {df}")
        # if i==5:
        if (i,j)==(3,5):
            plt.legend(title="# bins", loc='center left', bbox_to_anchor=(1.2, 1))
        if j==0:
            plt.ylabel(r"$\Delta$TMI")
        if j==5:
            ax2 = plt.twinx()
            ax2.set_yticks([])
            ax2.set_ylabel(f"r: 0.{r}")

ax[0,0].set_ylim((-.1,1.1))
plt.xscale("log")
plt.subplots_adjust(wspace=0.0, hspace=0.0)
plt.show()


In [ ]:
fig, ax = plt.subplots(6,6, sharex=True, sharey=True, figsize=(16,16), squeeze=False)
for i,r in enumerate([0,1,3,5,7,9]):
    for j,N in enumerate([30, 100, 300, 1000, 3000, 10000]):
        plt.sca(ax[i,j])
        for bin in [2,6,18,54,162]:
            x = []
            y = []
            yu = []
            yd = []
            for df in range(2,8):
                x.append(df)
                tmp = knn_values[N][df][r][bin]
                # tmp = shadow_c[N][df][r][bin]
                tmp2 = (tmp[:,0]-np.mean(tmp[:,1:],1))/(tmp[:,0]+1e-4)
                d,m,u = np.quantile(tmp2,[.25,.50,.75])
                y.append(m)
                yu.append(u-m)
                yd.append(m-d)
            p=plt.errorbar(x,y,[yu,yd],fmt="-^", elinewidth=1.5, capsize=3, label=bin, alpha=1 if bin<30 else 0.4)
        plt.plot([2,7], [0,0],":k",lw=2,zorder=3)
        if i==5:
            plt.xlabel("df")
            plt.xticks([2, 4, 6])
        if i==0:
            plt.title(f"N: {N}")
        # if i==5:
        if (i,j)==(3,6):
            plt.legend(title="# bins", loc='center left', bbox_to_anchor=(1.2, 1))
        if j==0:
            plt.ylabel(r"$\Delta$TMI")
        if j==6:
            ax2 = plt.twinx()
            ax2.set_yticks([])
            ax2.set_ylabel(f"r: 0.{r}")
# ax[0,0].set_ylim((-.2,0.2))
ax[0,0].set_ylim((-.1,1.1))
plt.subplots_adjust(wspace=0.0, hspace=0.0)
plt.show()


### Corrected

In [ ]:
with open("knn_verify30-30000.pickle", "rb") as fp:
    previous = pickle.load(fp)
    knn_values_c = {int(N):{int(df):{int(r):{int(bin):previous["values"][N][df][r][bin] for bin in previous["values"][N][df][r]} for r in previous["values"][N][df]} for df in previous["values"][N]} for N in previous["values"]}
    knn_shadow_c = {int(N):{int(df):{int(r):{int(bin):previous["shadow"][N][df][r][bin] for bin in previous["shadow"][N][df][r]} for r in previous["shadow"][N][df]} for df in previous["shadow"][N]} for N in previous["shadow"]}
    knn_correl = {int(N):{int(df):{int(r):{int(bin):previous["correl"][N][df][r][bin] for bin in previous["correl"][N][df][r]} for r in previous["correl"][N][df]} for df in previous["correl"][N]} for N in previous["correl"]}
    del previous

In [ ]:
fig, ax = plt.subplots(6,6, sharex=True, sharey=True, figsize=(16,16), squeeze=False)
for i,r in enumerate([0,1,3,5,7,9]):
    for j,df in enumerate(range(2,8)):
        plt.sca(ax[i,j])
        for bin in [2,6,18,54,162]:
            x = []
            y = []
            yu = []
            yd = []
            for N in [30, 100, 300, 1000, 3000, 10000]:
                x.append(N)
                tmp = knn_values_c[N][df][r][bin]
                # tmp = shadow_c[N][df][r][bin]
                tmp2 = 2*(tmp[:,0]-np.mean(tmp[:,1:],1))/(tmp[:,0]+np.mean(tmp[:,1:],1))#tmp[:,0]-np.mean(tmp[:,1:],1)
                d,m,u = np.quantile(tmp2,[.25,.50,.75])
                y.append(m)
                yu.append(u-m)
                yd.append(m-d)
            p=plt.errorbar(x,y,[yu,yd],fmt="-^", elinewidth=1.5, capsize=3, label=bin)
        plt.plot([30,10000], [0,0],":k",lw=2,zorder=3)
        plt.xlabel("N")
        if i==0:
            plt.title(f"df: {df}")
        # if i==5:
        if (i,j)==(3,5):
            plt.legend(title="# bins", loc='center left', bbox_to_anchor=(1.2, 1))
        if j==0:
            plt.ylabel(r"$\Delta$TMI")
        if j==5:
            ax2 = plt.twinx()
            ax2.set_yticks([])
            ax2.set_ylabel(f"r: 0.{r}")

ax[0,0].set_ylim((-.2,2.1))
plt.xscale("log")
plt.subplots_adjust(wspace=0.0, hspace=0.0)
plt.show()


In [ ]:
fig, ax = plt.subplots(6,6, sharex=True, sharey=True, figsize=(16,16), squeeze=False)
for i,r in enumerate([0,1,3,5,7,9]):
    for j,N in enumerate([30, 100, 300, 1000, 3000, 10000]):
        plt.sca(ax[i,j])
        for bin in [2,6,18,54,162]:
            x = []
            y = []
            yu = []
            yd = []
            if bin>N:
                continue
            for df in range(2,8):
                x.append(df)
                tmp = knn_values_c[N][df][r][bin]
                # tmp = shadow_c[N][df][r][bin]
                tmp2 = 2*(tmp[:,0]-np.mean(tmp[:,1:],1))/(tmp[:,0]+np.mean(tmp[:,1:],1))
                d,m,u = np.quantile(tmp2,[.25,.50,.75])
                y.append(m)
                yu.append(u-m)
                yd.append(m-d)
            p=plt.errorbar(x,y,[yu,yd],fmt="-^", elinewidth=1.5, capsize=3, label=bin, alpha=1 if bin<30 else 0.4)
        plt.plot([2,7], [0,0],":k",lw=2,zorder=3)
        if i==5:
            plt.xlabel("df")
            plt.xticks([2, 4, 6])
        if i==0:
            plt.title(f"N: {N}")
        # if i==5:
        if (i,j)==(3,5):
            plt.legend(title="# bins", loc='center left', bbox_to_anchor=(1.2, 1))
        if j==0:
            plt.ylabel(r"$\Delta$TMI")
        if j==5:
            ax2 = plt.twinx()
            ax2.set_yticks([])
            ax2.set_ylabel(f"r: 0.{r}")
# ax[0,0].set_ylim((-.2,0.2))
ax[0,0].set_ylim((-.2,2.1))
plt.subplots_adjust(wspace=0.0, hspace=0.0)
plt.show()


In [ ]:
corrector = Corrector(200, "/mnt/DATA/NonLinearMI/crafted_30000_2_r0_1000_bin32",30000, 1000, 32)
corrector.compute_correction()

In [ ]:
r=5
df=5
rho = r/10
mat=loadmat(f"/mnt/DATA/NonLinearMI/verify/crafted_30000_{df}_r{r}.mat")["crafted"][:1000,:,:]

In [ ]:
fix, ax = plt.subplots(1,3,sharex=True, sharey=True, squeeze=True)
plt.sca(ax[0])
plt.hist2d(*mat[:,:,0].T, bins=40, density=True)
plt.gca().set_aspect('equal', 'box')
plt.title("Raw")

plt.sca(ax[1])
norm = np.zeros((1000,2))
norm[:,0]=normalise(mat[:,0,0])
norm[:,1]=normalise(mat[:,1,0])
plt.hist2d(*norm.T, bins=40, density=True)
pmi = pair_mutual_information(*norm.T,32)
plt.gca().set_aspect('equal', 'box')
plt.title(f"Norm {corrector.correctI(pmi):.4}")

plt.sca(ax[2])
surr = surrogate(norm, True)
plt.hist2d(*surr.T, bins=40, density=True)
plt.gca().set_aspect('equal', 'box')
smi = pair_mutual_information(*surr.T,32)
plt.title(f"Surr {corrector.correctI(smi):.4}")
plt.show()

print(pmi, corrector.correctI(pmi),-0.5*np.log(1-rho**2))
print(smi, corrector.correctI(smi),-0.5*np.log(1-rho**2))

In [ ]:
plt.hist(knn_values[3000][3][5][16][0,1:], bins="auto",density=True)
yl = plt.ylim()
plt.plot([knn_values[3000][3][5][16][0,0]]*2, yl,":k")
plt.ylim(yl)
plt.title(knn_values[3000][3][5][16][0,0])
plt.show()

In [ ]:
corrector = Corrector(200, "/mnt/DATA/NonLinearMI/crafted_30000_3_r5_3000_bin16",30000, 1000, 16)
corrector.compute_correction()
df = 3
r = 5
mat=loadmat(f"/mnt/DATA/NonLinearMI/verify/crafted_30000_{df}_r{r}.mat")["crafted"][:3000,:,:]
# norm = np.zeros((3000,2))
# norm[:,0]=normalise(mat[:,0,0])
# norm[:,1]=normalise(mat[:,1,0])
# pmi = pair_mutual_information(*norm.T,16)#corrector.correctI()

smi = []
for ipat in task_producer(mat[:,:,0],99,True):
    smi.append(float(total_mutual_information(ipat,16)))#corrector.correctI()
plt.hist(smi[1:], bins="auto",density=True)
yl = plt.ylim()
pmi=smi[0]
plt.plot([pmi]*2, yl,":k")
plt.ylim(yl)
plt.title(pmi)
plt.show()

In [ ]:
from support import normalise, CMIknn
import numpy as np
import multiprocessing as mp
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
pts = 20
results = np.zeros((pts,100))
knn_estimator = CMIknn(32)
with mp.Pool(12) as pool:
    for r1 in tqdm(range(pts)):
        for j in range(100):
            x=np.random.multivariate_normal([0,0], [[1, r1/pts],[r1/pts,1]],400).T
            x[0,:]=normalise(x[0,:])
            x[1,:]=normalise(x[1,:])
            results[r1,j]=knn_estimator.get_MI(x)

plt.errorbar(-0.5*np.log(1-np.arange(20)**2/400), np.mean(results,1),np.std(results,1))
plt.plot([0,1.2],[0,1.2], ":k", lw=1)
plt.show()

In [ ]:
np.load("/mnt/DATA/NonLinearMI/knn_crafted_30000_2_r0_30_bin54/knn_patient00_54.npy")

# Approximate maps

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

parti = []
# todo = []
for N in [30,100,300,1000,3000,10000,30000]:
    for b in [2,6,18,54,162]:
        if (N,b) in todo:
            pass
        if N>b and os.path.isfile(f"/mnt/DATA/NonLinearMI/knn_crafted_30000_2_r0_{N}_bin{b}/newco_{b}.npy"):
            newco = np.load(f"/mnt/DATA/NonLinearMI/knn_crafted_30000_2_r0_{N}_bin{b}/newco_{b}.npy")
            trueval = np.load(f"/mnt/DATA/NonLinearMI/knn_crafted_30000_2_r0_{N}_bin{b}/trueval_{b}.npy")
            parti.append(pd.DataFrame({"nc":newco,"tv":trueval,"N":np.full_like(newco,N),"b":np.full_like(newco,b),"r":np.full_like(newco,(b)/N)}))
        else:
            if N>b and (N,b)!=(3000,6):
                pass
                # todo.append((N,b))
parti_df = pd.concat(parti).set_index(["N","b","tv"])

In [ ]:
from matplotlib.colors import LogNorm
fig = plt.figure(figsize=(16,12))
sns.lineplot(parti_df.reset_index(), x="tv",y="nc",hue="r",errorbar='sd', hue_norm=LogNorm())
plt.plot([0,2.4],[0,2.4],":g")
# plt.xlim(left=1.5)
# plt.ylim(bottom=1.5)
plt.show()

In [ ]:
parti_df.loc[(slice(None),slice(None),0)]

In [ ]:
available = np.sort(parti_df.r.unique())
def scegli (sr, val):
    sr1 = sr.loc[[i for i in sr.index if i[:2] not in todo]]
    print("In scegli:")
    display(sr1)
    if sum(sr1==val)>1:
        tsr = sr1[sr1==val]
        ind = tsr.loc[(slice(None),slice(max(tsr.index.get_level_values(1)),None),0)].index
    else:
        ind = sr1[sr1==val].index
        print("just one")
    print(ind)
    return ind

for miss in todo:
    print(miss)
    N,b = miss
    i = np.searchsorted(available, b/N)
    print(available[i-1:i+1], b/N)
    display(parti_df[parti_df.r.isin(available[i-1:i+1])].loc[(slice(None),slice(None),0)])
    other = parti_df[(parti_df.r.isin(available[i-1:i+1]))].loc[(slice(None),slice(None),0),"r"]
    
    minInd = scegli(other, min(other))
    maxInd = scegli(other, max(other))
    minLog = np.log10(other.loc[minInd]).values
    maxLog = np.log10(other.loc[maxInd]).values
    rLog = np.log10(b/N)

    minWeight = (rLog-minLog)/(maxLog-minLog)
    maxWeight = (maxLog-rLog)/(maxLog-minLog)
    print("Pesi:",minWeight, maxWeight)

    if not os.path.isdir(f"/mnt/DATA/NonLinearMI/knn_crafted_30000_2_r0_{N}_bin{b}/"):
        os.mkdir(f"/mnt/DATA/NonLinearMI/knn_crafted_30000_2_r0_{N}_bin{b}/")
    np.save(f"/mnt/DATA/NonLinearMI/knn_crafted_30000_2_r0_{N}_bin{b}/trueval_{b}.npy",trueval)
    np.save(f"/mnt/DATA/NonLinearMI/knn_crafted_30000_2_r0_{N}_bin{b}/newco_{b}.npy",(parti_df.loc[minInd[0][:2],"nc"]*maxWeight+parti_df.loc[maxInd[0][:2],"nc"]*minWeight).values)

In [ ]:
other.loc[[i for i in other.index if i[:2] not in todo]]

In [ ]:
plt.plot([2,6,18,54,162],[5.5,6.4,9.3,12.3,25.3])
plt.plot([1,100],[6.28,20])
plt.xscale("log")
plt.yscale("log")